# Lab 5 · Bounded tool execution with Gemini
## Goal
Extend the manual ReAct and native tool work in AAI[sum26] Lab 3. Implement strict dispatch, preserve tool-call identity, and finish a model→tool→model cycle. Prerequisites: Python dictionaries, exceptions, Lab 3. Estimated time: 100 minutes (15 setup, 45 implementation, 25 tests, 15 report).

By the end, distinguish a well-formed tool request from an authorized action. Use an explicit operation schema instead of evaluating arbitrary arithmetic expressions.

## Setup
Use Python 3.11+ and the supplied `requirements.txt` in a fresh environment; select that environment as the notebook kernel. See `README.md` for local and Colab setup. These notebooks are self-contained once dependencies are installed.

Default execution runs worked examples with synthetic fixtures, without API requests. Complete TODO functions, set `RUN_EXERCISES=True`, and rerun from the first cell. Instructor copies contain solutions and enable the exercise checks. Live sections require `GEMINI_API_KEY` and `GEMINI_MODEL`; never paste keys into saved cells or outputs. Model selection is explicit because availability changes. No OpenAI API key is required for the Gemini path.

In [ ]:
import os, json, math, time
RUN_EXERCISES = False  # Set True after completing the TODO functions.
RUN_LIVE = False       # Explicit opt-in: live requests use your Gemini quota.
MODEL = os.getenv("GEMINI_MODEL", "")  # Select a model available in your account.

def live_config():
    if not MODEL or not os.getenv("GEMINI_API_KEY"):
        raise RuntimeError("Set GEMINI_MODEL and GEMINI_API_KEY before RUN_LIVE=True")
    return MODEL, os.environ["GEMINI_API_KEY"]

print("Mode: offline fixtures; these are not model performance measurements.")

## Steps
### 1. Inspect a bounded tool
This worked function permits two operations on bounded finite numbers. Booleans are rejected even though Python treats them as integers. The bound is a lab policy, not a universal numeric limit.

In [ ]:
def arithmetic(a: float, b: float, operation: str) -> dict:
    """Apply add or divide to two finite numbers of magnitude at most one million."""
    if any(type(x) not in (int, float) or abs(x)>1_000_000 or not math.isfinite(x) for x in (a,b)):
        return {"error": "invalid_number"}
    if operation not in ("add", "divide"):
        return {"error": "unknown_operation"}
    if operation == "divide" and b == 0:
        return {"error": "division_by_zero"}
    value = a+b if operation == "add" else a/b
    return {"value":value} if math.isfinite(value) else {"error":"nonfinite_result"}

assert arithmetic(20,22,"add") == {"value":42}
assert arithmetic(1,0,"divide") == {"error":"division_by_zero"}
print(arithmetic(20,22,"add"))

### 2. TODO: implement the dispatch boundary (35 points)
`dispatch(name, args, registry)` must use the supplied registry. Allow only keys `a`, `b`, `operation`; require a dictionary with exactly those keys. Return structured errors for unknown tools and bad arguments. Do not call a tool on rejected input. Catch tool exceptions as `tool_failure` without returning exception text.

In [ ]:
def dispatch(name, args, registry):
    raise NotImplementedError("TODO: validate and dispatch")

In [ ]:
if RUN_EXERCISES:
    args = {"a":20,"b":22,"operation":"add"}
    assert dispatch("arithmetic", args, {"arithmetic":arithmetic}) == {"value":42}
    assert dispatch("arithmetic", args, {"arithmetic":lambda **kw: {"value":99}}) == {"value":99}
    assert dispatch("missing", args, {}) == {"error":"unknown_tool"}
    for bad in [None, [], {}, {**args,"extra":1}, {**args,"a":True}, {**args,"a":float("nan")}, {**args,"b":float("inf")}, {**args,"a":10**7}]:
        assert dispatch("arithmetic", bad, {"arithmetic":arithmetic}) == {"error":"invalid_arguments"}
    def broken(**kw): raise RuntimeError("private detail")
    assert dispatch("arithmetic", args, {"arithmetic":broken}) == {"error":"tool_failure"}
    print("Dispatch checks passed")

### 3. Complete the native protocol
The manual loop below disables automatic execution, appends the model's complete content, and sends function responses back. Preserving the complete model content retains provider metadata rather than reconstructing only text. It caps both rounds and executed calls. Multiple calls are dispatched sequentially. A round is one model request; the call cap is a separate constraint.

Read the [Gemini function-calling documentation](https://ai.google.dev/gemini-api/docs/function-calling).

In [ ]:
from google.genai import types

def native_loop(client, model, task, dispatcher, max_rounds=4, max_calls=4):
    if type(max_rounds) is not int or not 1 <= max_rounds <= 8:
        raise ValueError("max_rounds must be an integer from 1 to 8")
    if type(max_calls) is not int or not 0 <= max_calls <= 8:
        raise ValueError("max_calls must be an integer from 0 to 8")
    history=[types.Content(role="user",parts=[types.Part(text=task)])]
    trace=[]
    for _ in range(max_rounds):
        response=client.models.generate_content(model=model, contents=history,
            config=types.GenerateContentConfig(tools=[arithmetic],
                automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
                max_output_tokens=1024))
        if not response.candidates or not response.candidates[0].content:
            return {"status":"empty_response", "trace":trace}
        content=response.candidates[0].content
        calls=[p.function_call for p in (content.parts or []) if p.function_call]
        if not calls:
            answer="".join(p.text or "" for p in (content.parts or []) if not p.thought).strip()
            return {"status":"done" if answer else "empty_response", "answer":answer,"trace":trace}
        if len(trace)+len(calls)>max_calls:
            return {"status":"call_budget", "trace":trace}
        history.append(content)
        replies=[]
        for call in calls:
            result=dispatcher(call.name,dict(call.args or {}),{"arithmetic":arithmetic})
            trace.append({"call_id":call.id,"tool":call.name,"result":result})
            replies.append(types.Part(function_response=types.FunctionResponse(
                name=call.name,id=call.id,response=result)))
        history.append(types.Content(role="user",parts=replies))
    return {"status":"round_budget","trace":trace}

## Checks
### 4. Verify the real protocol objects without a network
The fake client checks that the second request actually contains the tool result and matching call ID. This tests orchestration, not Gemini behavior.

In [ ]:
from types import SimpleNamespace
class FakeGemini:
    def __init__(self): self.models=self; self.calls=0
    def generate_content(self, **kw):
        self.calls += 1
        if self.calls == 1:
            part=types.Part(function_call=types.FunctionCall(name="arithmetic",id="call-1",
                args={"a":20,"b":22,"operation":"add"}))
        else:
            result=kw["contents"][-1].parts[0].function_response
            assert result.id == "call-1" and result.response == {"value":42}
            part=types.Part(text="42")
        return SimpleNamespace(candidates=[SimpleNamespace(content=types.Content(role="model",parts=[part]))])

# Narrow worked dispatcher; replace with your validated boundary in the exercise check.
def worked_dispatch(name,args,registry):
    return registry[name](**args)
assert native_loop(FakeGemini(),"fixture","add",worked_dispatch)["answer"] == "42"
assert native_loop(FakeGemini(),"fixture","add",worked_dispatch,max_calls=0)["status"] == "call_budget"
assert native_loop(FakeGemini(),"fixture","add",worked_dispatch,max_rounds=1)["status"] == "round_budget"
if RUN_EXERCISES:
    assert native_loop(FakeGemini(),"fixture","add",dispatch)["answer"] == "42"
print("Protocol checks passed")

### 5. Optional live run
Complete the dispatch exercise first. No retries are hidden in this harness; an API failure should remain visible. Client timeout is 20 seconds per request, and the loop has at most four requests.

In [ ]:
if RUN_LIVE:
    if not RUN_EXERCISES: raise RuntimeError("Complete and enable exercise checks first")
    from google import genai
    model,key=live_config()
    with genai.Client(api_key=key,http_options=types.HttpOptions(timeout=20_000,retry_options=types.HttpRetryOptions(attempts=1))) as client:
        print(native_loop(client,model,"Use arithmetic to add 20 and 22.",dispatch))
else:
    print("Live Gemini check skipped")

## Next Steps · Student submission
1. Dispatch implementation and passing checks (35 points).
2. Add tests for missing candidates, two tool calls in one response, malformed arguments, and a tool that raises (25 points).
3. Explain why AST whitelisting alone does not bound computational cost; explain round vs tool budgets (20 points).
4. Report a live trace if available, or clearly label offline-only evidence; explain what remains untested (20 points).

Answer here. Do not submit API keys. Native tool support does not remove authorization or validation responsibilities. This exercise extends the incomplete manual round-trip in the reference Lab 3; it does not reproduce a research benchmark.